# 04 · Thinking off or thinking with a budget (stage S2)

Every run now sends the system prompt and asks for the reply `ANSWER: <answer>`. The chat template cannot switch thinking off
(`reasoning_effort="none"` does not stop it), so two ways of getting the answer are compared:

- **P** (prefill, no thinking): the answer message is opened for the model by appending it to the prompt.
- **T** (thinking with a budget): the model thinks first; after `think_budget` tokens the answer message is opened for it.

Done-when: the chosen variant parses at least 95% over the GQA and TextVQA dev items together. The study is small (20 items per
benchmark), so accuracy differences of a few points are noise.

**How to run:** Run All, no edits. It needs the GPUs of `n_copies` in `configs/mvp.yaml` (2 copies: GPUs 0-3 all free). Step 5
(res_px) is optional and off (`RUN_RES_PX`). T is the slow part: allow most of an hour.
Shut down the kernels of other notebooks first: they keep their GPUs.

In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
# GPU choice must precede CUDA init, which importing adaloop.api already does: the one import outside adaloop.api
from utils.gpus import select_for_notebook

CONFIG = "configs/mvp.yaml"   # the config this notebook passes to setup_notebook
# GPUs for the config's n_copies (2: every gpu group, 1: the first free pair); changing it needs a kernel restart
GPU_IDS, N_COPIES = select_for_notebook(CONFIG)

In [ ]:
import math
import os
import socket

In [ ]:
from adaloop import api

ROOT, cfg = api.setup_notebook("configs/mvp.yaml")
print("ROOT", ROOT, "| config_hash", api.config_hash(cfg), "| git", api.git_hash())

In [ ]:
# torch is imported here because the GPU list needs it; the other third-party imports come after this cell
import torch

IS_INARA = "inara" in socket.gethostname()
print("host", socket.gethostname(), "| on Inara:", IS_INARA)
print("chosen GPUs, physical ids CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        total_gib = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"chosen GPU {i} (torch index) | {torch.cuda.get_device_name(i)} | {total_gib:.0f} GiB")
else:
    print("no CUDA GPU visible: the model steps will fail, the rest of the notebook can still be read")


In [ ]:
import pandas as pd


In [ ]:
# Settings
N_VARIANT_ITEMS = cfg["sample_items"]  # dev items per benchmark in step 4
R = 3  # loop passes for every evaluation here
BATCH_SIZE = 1  # the reference path
N_COPY_CHECK = 4  # step 3: items compared between one copy and all copies
PREFILL_P = cfg["no_think_prefill"]  # opens the answer message and writes "ANSWER:"
VARIANTS = {  # config keys each variant changes
    "P": {"answer_prefill": PREFILL_P},
    "T": {"answer_prefill": None, "reasoning_effort": cfg["reasoning_effort"], "think_budget": cfg["think_budget"]},
}
RUN_VARIANTS = True  # step 4: N_VARIANT_ITEMS items x 2 benchmarks x 2 variants
# The variant used in steps 5 and 6. Step 4 fills it in; set "P" or "T" by hand when RUN_VARIANTS is False.
VARIANT = None
RUN_RES_PX = False  # step 5: N_RES TextVQA items x 2 sizes
N_RES = cfg["sample_items"]  # TextVQA items in step 5
RES = [768, 1024]  # long-side sizes compared in step 5
VARIANTS

### Step-1:- Load the model

The model and its processor are loaded once and reused by every step below. Loading is the slow part of the notebook.

In [ ]:
print("[STAGE] load model")
vlms = api.load_models(cfg, N_COPIES)
vlm = vlms[0]
tok = vlm.processor.tokenizer

### Step-2:- The prompt the model sees

One GQA dev item, rendered back from its input ids, to check where the system message lands. P appends the answer marker at the
end; T stops after the user turn. The repeated image placeholder is shown once.

In [ ]:
item = api.load_gqa(cfg, "dev", limit=1, root=ROOT)[0]
item.question, item.answers

In [ ]:
inputs_p = api.item_inputs(vlm, item, {**cfg, **VARIANTS["P"]})
ids = inputs_p["input_ids"][0].tolist()
image_id = vlm.model.config.image_token_id
# keep only the first of each run of image placeholders
shown = [t for i, t in enumerate(ids) if not (t == image_id and i and ids[i - 1] == image_id)]
print(tok.decode(shown, skip_special_tokens=False))
print("prompt tokens", len(ids), "| image placeholders", ids.count(image_id))

In [ ]:
# each marker must be one token, or the prefill would not be what the model itself emits
prefill_ids = tok(PREFILL_P, add_special_tokens=False)["input_ids"]
assert tok.decode(prefill_ids, skip_special_tokens=False) == PREFILL_P, prefill_ids
for marker in ("<|message_model|>", "<|content_text|>"):
    assert len(tok(marker, add_special_tokens=False)["input_ids"]) == 1, f"{marker} is not one token"
assert inputs_p["input_ids"][0, -len(prefill_ids):].tolist() == prefill_ids, "the prompt of P does not end with the prefill"
print(f"✓ prompt: P ends with the prefill ({len(prefill_ids)} tokens)")

### Step-3:- One item under both variants

The extraction rule is checked on hand-written outputs first, then one item runs under P and T before any batch.

**T is slow:** about 4 tokens/s at batch 1, and up to 288 tokens per item.

In [ ]:
OPEN, END = "<|content_text|>", "<|end_message|>"
cases = [  # (raw output, hit_limit, prefilled, expected answer)
    (f"ANSWER: blue{END}", False, True, "blue"),
    (f"blue{END}", False, True, None),  # no marker: no answer
    ("ANSWER: woman\n\nWe need answer", True, True, "woman"),  # the answer ends at the newline
    ("ANSWER: a long answer that got cut", True, True, None),
    # thinking, then the answer message: an ANSWER line inside thinking is ignored
    (f"<|content_thinking|>maybe\nANSWER: stop{END}<|message_model|>{OPEN}ANSWER: go{END}", False, False, "go"),
]
for raw, hit, prefilled, want in cases:
    got = api.extract_answer(raw, hit, prefilled=prefilled, answer_format=cfg["answer_format"])
    assert got == want, f"extract_answer({raw[-40:]!r}, hit_limit={hit}, prefilled={prefilled}) gave {got!r}, want {want!r}"
print("✓ extraction on fixed outputs")

# P through the read path of run_evaluation: the prefill already wrote "ANSWER:", so the model writes only the answer
def reply(raw):
    """A finished generation whose output is `raw`."""
    return api.GenerationResult(token_ids=torch.tensor([]), text=raw, raw_text=raw, n_input_tokens=0, n_visual_tokens=0,
                                hit_limit=False, ttft_s=None, generate_s=0.0, peak_mem_gb=None, n_think_tokens=0,
                                n_answer_tokens=0)


p_cfg = {**cfg, **VARIANTS["P"]}
assert PREFILL_P.endswith(OPEN + cfg["answer_format"])
assert api.read_answer(reply(f" red{END}"), p_cfg) == "red"
assert api.read_answer(reply(f"yes{END}"), p_cfg) == "yes"
assert api.read_answer(reply(f"{END}"), p_cfg) is None  # empty answer
assert api.read_answer(reply(f"ANSWER: red{END}"), {**cfg, **VARIANTS["T"]}) == "red"
assert api.read_answer(reply(f"red{END}"), {**cfg, **VARIANTS["T"]}) is None  # thinking on: strict, no marker, no answer
print("✓ P reply: read behind the prefilled ANSWER: marker")

In [ ]:
print("[STAGE] one item")
one = []
for variant, changes in VARIANTS.items():
    row = api.evaluate_item(vlm, item, {**cfg, **changes}, R)
    one.append({"variant": variant, **{k: row[k] for k in ("pred", "correct", "correct_lmms", "format_ok",
                "n_think_tokens", "n_answer_tokens", "forced", "hit_limit", "generate_s")}})
one = pd.DataFrame(one)
one

In [ ]:
assert list(one.variant) == list(VARIANTS)
assert one.set_index("variant").loc["P", "n_think_tokens"] == 0, "P produced thinking tokens"
print("✓ one item: both variants ran, P did not think")

#### One copy against all copies

Several copies run the items in threads. They must give the same answers as one copy, or the batch run below cannot be trusted.
Run under P, which is fast; skipped with one copy.

In [ ]:
if N_COPIES > 1:
    print("[STAGE] copies")
    half = N_COPY_CHECK // 2
    copy_items = api.load_gqa(cfg, "dev", limit=half, root=ROOT) + api.load_textvqa(cfg, "dev", limit=half, root=ROOT)
    copies_table = pd.DataFrame(api.compare_copies(vlms, copy_items, p_cfg, R))
copies_table if N_COPIES > 1 else "skipped: N_COPIES is 1"

In [ ]:
if N_COPIES > 1:
    assert len(copies_table) == N_COPY_CHECK and copies_table.same_pred.all(), "copies changed an answer; see the table"
    print(f"✓ copies: {N_COPIES} copies give the answers of one copy on {N_COPY_CHECK} items")
else:
    print("skipped: N_COPIES is 1")

### Step-4:- P against T (done-when)

The batch run: N_VARIANT_ITEMS GQA and N_VARIANT_ITEMS TextVQA dev items per variant. The parse rate over both benchmarks
together picks the variant (accuracy breaks a tie), and the done-when is that it reaches 95% (DEC3 S2). Rates and accuracies are
percentages; `accuracy` is the headline score (GQA official match, TextVQA VQA accuracy), `accuracy_lmms` the lmms-eval one
(equal for TextVQA). With 2 copies `s_per_item` includes contention between them.

In [ ]:
# One evaluation with an in-memory config copy; the file name ends with the hash of that copy
def run(bench, bench_items, variant, res_px):
    """One evaluation of `bench_items` with `variant` at `res_px`; return (summary, rows)."""
    run_cfg = {**cfg, **VARIANTS[variant], "res_px": res_px}
    name = f"{bench}_dev_n{len(bench_items)}_res{res_px}_R{R}_{variant}_b{BATCH_SIZE}_{api.config_hash(run_cfg)}.jsonl"
    out = ROOT / cfg["out_dir"] / "nb04" / name
    summary = api.run_evaluation(None, bench_items, run_cfg, R, out, resume=True, vlms=vlms, batch_size=BATCH_SIZE)
    return summary, api.read_results(out)

In [ ]:
if RUN_VARIANTS:
    print("[STAGE] variants")
    items = {
        "gqa": api.load_gqa(cfg, "dev", limit=N_VARIANT_ITEMS, root=ROOT),
        "textvqa": api.load_textvqa(cfg, "dev", limit=N_VARIANT_ITEMS, root=ROOT),
    }
    table, variant_rows = [], {}
    for variant in VARIANTS:
        for bench, bench_items in items.items():
            summary, rows = run(bench, bench_items, variant, cfg["res_px"])
            variant_rows[variant, bench] = rows
            table.append({"variant": variant, "benchmark": bench, "n": summary["n"], "accuracy": summary["accuracy"],
                          "accuracy_lmms": summary["accuracy_lmms"], "format_ok": summary["format_ok_rate"],
                          "parse_rate": summary["parse_rate"], "answer_len_ok": summary["answer_len_ok_rate"],
                          "n_think_tokens": summary["n_think_tokens_mean"], "forced": summary["forced_rate"],
                          "s_per_item": summary["generate_s_mean"]})
    table = pd.DataFrame(table)
table if RUN_VARIANTS else "skipped: RUN_VARIANTS is False"

#### How long does T think?

A forced item stops at `think_budget`, so a median close to the budget means most items are cut off.

In [ ]:
if RUN_VARIANTS:
    think = pd.DataFrame([r for (variant, _), rows in variant_rows.items() if variant == "T" for r in rows])
think.groupby("benchmark").n_think_tokens.describe() if RUN_VARIANTS else "skipped: RUN_VARIANTS is False"

#### Answers and failures

Per item: the question, each variant's parsed answer and the references. Then at most 5 outputs that gave no answer.

In [ ]:
if RUN_VARIANTS:
    per_item = pd.DataFrame([
        {"benchmark": bench, "question": it.question,
         "pred_P": variant_rows["P", bench][i]["pred"], "pred_T": variant_rows["T", bench][i]["pred"],
         "references": sorted(set(it.answers))}
        for bench, bench_items in items.items() for i, it in enumerate(bench_items)
    ])
per_item.groupby("benchmark").head(5) if RUN_VARIANTS else "skipped: RUN_VARIANTS is False"

In [ ]:
if RUN_VARIANTS:
    failures = [(v, b, r) for (v, b), rows in variant_rows.items() for r in rows if not r["parse_ok"]]
    print(f"{len(failures)} unparsed outputs; showing the last 150 chars of at most 5")
    for variant, bench, r in failures[:5]:
        print(f"  {variant} {bench} {r['item_id']}: {r['raw_output'][-150:]!r}")
else:
    print("skipped: RUN_VARIANTS is False")

In [ ]:
# Choose by parse rate over both benchmarks together, then by accuracy
if RUN_VARIANTS:
    all_rows = pd.DataFrame([{"variant": v, "parse_rate": 100.0 * r["parse_ok"], "accuracy": 100.0 * r["correct"]}
                             for (v, _), rows in variant_rows.items() for r in rows])
    combined = all_rows.groupby("variant")[["parse_rate", "accuracy"]].mean()
    VARIANT = combined.sort_values(["parse_rate", "accuracy"], ascending=False).index[0]
    print("chosen variant", VARIANT)
else:
    assert VARIANT in VARIANTS, "RUN_VARIANTS is False: set VARIANT to 'P' or 'T' in the settings cell"
    print("skipped: RUN_VARIANTS is False; using VARIANT", VARIANT, "from the settings")
combined if RUN_VARIANTS else None

In [ ]:
if RUN_VARIANTS:
    assert all(len(rows) == N_VARIANT_ITEMS for rows in variant_rows.values()), [len(r) for r in variant_rows.values()]
    parse = combined.loc[VARIANT, "parse_rate"]
    assert parse >= 95.0, f"{VARIANT}: combined parse rate {parse:.1f} is below 95"
    print(f"✓ variants: {VARIANT} parses {parse:.1f}% over GQA + TextVQA")
else:
    print("skipped: RUN_VARIANTS is False")

### Step-5:- res_px: 768 vs 1024 on TextVQA dev (optional)

Off by default (`RUN_RES_PX`). `resize_long_side` never upscales, so an image whose long side is already below `res_px` is sent as
it is. The processor makes one visual token per patch, so the expected token count is a check on the resize.

In [ ]:
if RUN_RES_PX:
    res_items = api.load_textvqa(cfg, "dev", limit=N_RES, root=ROOT)
    long_sides = [max(it.image.size) for it in res_items]
    print("original long side: min", min(long_sides), "max", max(long_sides))
    patch = vlm.processor.image_processor.size
    print("patch", patch)

    expected = {}
    for res_px in RES:
        sizes = [api.resize_long_side(it.image, res_px).size for it in res_items]
        expected[res_px] = sum(math.ceil(w / patch["width"]) * math.ceil(h / patch["height"]) for w, h in sizes) / len(sizes)
        print(f"res_px {res_px}: {sum(s < res_px for s in long_sides)}/{len(res_items)} images smaller than res_px, "
              f"largest sent {max(max(s) for s in sizes)}, expected mean n_vis {expected[res_px]:.1f}")
else:
    print("skipped: RUN_RES_PX is False")

In [ ]:
res_table = None  # stays None without the comparison; F02 then has two panels
if RUN_RES_PX:
    assert VARIANT in VARIANTS, "set VARIANT to 'P' or 'T' (step 4 does this, or set it in the settings)"
    print("[STAGE] res_px")
    res_table = []
    for res_px in RES:
        summary, rows = run("textvqa", res_items, VARIANT, res_px)
        res_table.append({"res_px": res_px, "variant": VARIANT, "n": len(rows), "accuracy": summary["accuracy"],
                          "parse_rate": summary["parse_rate"], "n_vis_mean": summary["n_vis_mean"],
                          "n_vis_expected": expected[res_px], "generate_s_mean": summary["generate_s_mean"],
                          "peak_mem_gb_max": summary["peak_mem_gb_max"]})
    res_table = pd.DataFrame(res_table)
res_table if RUN_RES_PX else "skipped: RUN_RES_PX is False"

In [ ]:
if RUN_RES_PX:
    assert (res_table.n == N_RES).all(), res_table.n.tolist()
    print("✓ res_px: every size has all N_RES rows")
else:
    print("skipped: RUN_RES_PX is False")


### Step-6:- Decision to log

Paste these lines into `docs/decisions.md`, fill in the blanks, then set `answer_prefill`, `reasoning_effort`, `think_budget` (and
`res_px` if it was compared) in `configs/mvp.yaml`.

In [ ]:
if RUN_VARIANTS:
    chosen = {**cfg, **VARIANTS[VARIANT]}
    print(f"## <date> · S2 · answer path (git {api.git_hash()})")
    print(f"- Decision: variant {VARIANT} (answer_prefill = {chosen['answer_prefill']!r}, reasoning_effort = "
          f"{chosen['reasoning_effort']}, think_budget = {chosen['think_budget'] if VARIANT == 'T' else 'off'}) because <reason>.")
    print(f"- Dev n={N_VARIANT_ITEMS} per benchmark, R={R}, res_px={cfg['res_px']}, batch {BATCH_SIZE}:")
    for r in table.itertuples():
        print(f"  - {r.variant} {r.benchmark}: acc {r.accuracy:.1f} (lmms {r.accuracy_lmms:.1f}), parse {r.parse_rate:.1f}, "
              f"format ok {r.format_ok:.1f}, forced {r.forced:.1f}, think tokens {r.n_think_tokens:.0f}, {r.s_per_item:.2f} s/item")
    if RUN_RES_PX:
        summary_text = "; ".join(f"{r.res_px}: acc {r.accuracy:.1f}, n_vis {r.n_vis_mean:.0f}, {r.generate_s_mean:.2f}s, "
                                 f"{r.peak_mem_gb_max:.1f} GB" for r in res_table.itertuples())
        print(f"- Decision: res_px = <768|1024> because <reason>. TextVQA dev n={N_RES}: {summary_text}.")
    else:
        print(f"- res_px stays {cfg['res_px']}: 768 vs 1024 not compared.")
else:
    print("skipped: this decision needs RUN_VARIANTS = True")

### Step-7:- Figure F02

(a) parse rate and (b) accuracy of P and T on GQA and TextVQA; with `RUN_RES_PX`, also (c) TextVQA accuracy at 768 vs 1024 px.
It shows the evidence behind the decision in step 6.

In [ ]:
if RUN_VARIANTS:
    api.set_paper_style()
    fig = api.plot_thinking_off(table, res_table)
else:
    print("skipped: RUN_VARIANTS must be True to plot F02")

In [ ]:
if RUN_VARIANTS:
    pdf_path, png_path = api.save_figure(fig, "F02_thinking_off", final=False)
else:
    print("skipped: F02 not saved")

In [ ]:
if RUN_VARIANTS:
    assert pdf_path.is_file() and png_path.is_file(), f"missing {pdf_path} or {png_path}"
    print("✓ F02_thinking_off draft saved as PDF and PNG (final=False, paper/ untouched)")
else:
    print("skipped: F02 not saved")

In [ ]:
print("✓ notebook 04 done | VARIANT", VARIANT)